# Kelmarsh 2022 — Stage 3 Local Clustering

Fit January–June; selection July–December reused in preprocessing; final refit is descriptive. All 60 frozen coordinate__ features are retained. Reconstruction of preprocessing is documented separately. Run from the directory containing kelmarsh_stage3.py. No interactive state from preprocessing is required.

## 1. Environment validation

Dependencies are local; threads are bounded. Versions and RAM are recorded.

In [1]:
from pathlib import Path
import sys, os, json, time
ROOT=Path.cwd()
assert (ROOT/'kelmarsh_stage3.py').is_file(), 'Run from the project directory'
sys.path.insert(0,str(ROOT/'.stage3_deps'))
sys.path.insert(0,str(ROOT))
os.environ.setdefault('OMP_NUM_THREADS','4')
from kelmarsh_stage3 import Experiment, synthetic_checks, digest
experiment=Experiment(ROOT)
experiment.environment()


## 2. Project and artifact discovery

Only original exported windows enter clustering. Raw inputs and implementation provenance are checked.

In [2]:
print('Workspace:', ROOT)
print('Preprocessing artifacts:', experiment.input_dir)
print('Raw SCADA files:', len(list(ROOT.glob('Turbine_Data_Kelmarsh_*_2022*.csv'))))
print('Preprocessing copy:', ROOT/'Kelmarsh_2022_Preprocessing_Local_Reconstructed.ipynb')


Workspace: C:\College\Sem5\DATMIN\Kelmarsh_SCADA_2022_4457
Preprocessing artifacts: C:\College\Sem5\DATMIN\Kelmarsh_SCADA_2022_4457\artifacts\kelmarsh_2022
Raw SCADA files: 6
Preprocessing copy: C:\College\Sem5\DATMIN\Kelmarsh_SCADA_2022_4457\Kelmarsh_2022_Preprocessing_Local_Reconstructed.ipynb


## 3. Synthetic technical checks

Synthetic data tests are separated from research results.

In [3]:
synthetic_checks()

Synthetic clustering and temporal checks passed; excluded from research evidence.


## 4–5. Input verification and feature matrix

SHA-256, schema, finite values, UTC, completeness, uniqueness, chronological split, original-unit statistics, and frozen scaler reconstruction are asserted.

In [4]:
experiment.load()

Verified (52241, 60); train=25881, selection=26360


## 6. Feature-space diagnosis before modeling

Variance, correlation, paired distance concentration, extreme observations and feature-family contributions. Frozen main features remain untouched.

In [5]:
from stage3_structural import StructuralStudy
structural=StructuralStudy(experiment)
structural.feature_diagnosis()
reuse_centroids=False if os.environ.get('KELMARSH_FORCE_RETRAIN')=='1' else structural.restore_centroid_cache()


{
  "sampled_pairs": 19999,
  "train_dimensions": 60,
  "distance_quantiles": {
    "0.01": 3.943567686994518,
    "0.1": 5.888239406041466,
    "0.5": 9.280805697538865,
    "0.9": 15.356750169709967,
    "0.99": 19.233101843392433
  },
  "distance_cv": 0.46704349699434416,
  "ordinary_pair_median": 9.187256954823402,
  "extreme_pair_median": 18.793141011538268,
  "upper_one_percent_norm_threshold": 15.508659661788318,
  "extreme_observations_n": 259,
  "top_one_percent_fraction_total_squared_norm": 0.11785527089196761,
  "rotor_generator_correlations": {
    "mean": 0.9999922348659946,
    "std": 0.9998761704891119,
    "min": 0.9999873770714585,
    "max": 0.9999621381627856,
    "endpoint_change": 0.9999330973675277
  },
  "preprocessing_altered": false,
  "main_input_sha256": "1165215e960379a4a17b5d28e9e8ac0f848b334738129f5c2625cf3e8942b94c"
}


Verified cache: all K-Means/FCM/Ward k=2–10 experiments already executed; restoring trained models and measured results.


## 7. K-Means++ sweep and internal validity

k=2–10; explicit n_init, max_iter, tolerance and seeds. Existing executed results may be reused only after checksums, parameters and feature order match; set KELMARSH_FORCE_RETRAIN=1 for new fits.

In [6]:
if not reuse_centroids: experiment.kmeans_sweep()
else: display(__import__('pandas').DataFrame(experiment.kmeans_rows))

,silhouette,dbi,chi,occupied_clusters,min_cluster_n,min_cluster_fraction,imbalance_ratio,very_small_cluster_warning,method,model,...,mean_strength_selection,normalized_entropy_selection,ambiguous_fraction_selection,iterations,best_seed,membership_uniformity_warning,entropy_after_persistence,ambiguity_after_persistence,entropy_after_transition,ambiguity_after_transition
0,0.329538,1.856048,4071.275276,2,5396,0.204704,3.885100,False,KMeans,KMeans_k2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.159640,1.753498,3303.269423,3,2727,0.103452,6.376604,False,KMeans,KMeans_k3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.177315,1.781616,3049.039324,4,1323,0.050190,11.942555,False,KMeans,KMeans_k4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.180857,1.484015,3635.878678,5,4,0.000152,3955.750000,True,KMeans,KMeans_k5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.125096,1.753561,3193.322600,6,4,0.000152,2911.500000,True,KMeans,KMeans_k6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0.135369,1.767890,2909.588900,7,4,0.000152,2917.750000,True,KMeans,KMeans_k7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,0.103539,1.908095,2661.287419,8,4,0.000152,2415.000000,True,KMeans,KMeans_k8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,0.114365,1.846789,2529.217558,9,4,0.000152,2396.250000,True,KMeans,KMeans_k9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.118481,1.813001,2356.466710,10,4,0.000152,2297.500000,True,KMeans,KMeans_k10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Fuzzy C-Means challenger

Full k=2–10, matching K-Means. Same features and period; five initializations for every k. Hard validity and fuzzy membership diagnostics remain distinct.

In [7]:
if not reuse_centroids: experiment.fcm_challenger()
else: display(__import__('pandas').DataFrame(experiment.fcm_rows))

,silhouette,dbi,chi,occupied_clusters,min_cluster_n,min_cluster_fraction,imbalance_ratio,very_small_cluster_warning,method,model,...,mean_strength_selection,normalized_entropy_selection,ambiguous_fraction_selection,iterations,best_seed,membership_uniformity_warning,entropy_after_persistence,ambiguity_after_persistence,entropy_after_transition,ambiguity_after_transition
0,0.120286,2.135437,3122.119143,2,10252,0.388923,1.571206,False,FCM,FCM_k2,...,0.500003,1.0,1.0,424.0,2024.0,True,1.0,1.0,1.0,1.0
1,0.120286,2.135437,3122.119143,2,0,0.000000,inf,True,FCM,FCM_k3,...,0.333336,1.0,1.0,464.0,2025.0,True,1.0,1.0,1.0,1.0
2,0.120286,2.135437,3122.119143,2,0,0.000000,inf,True,FCM,FCM_k4,...,0.250003,1.0,1.0,467.0,2025.0,True,1.0,1.0,1.0,1.0
3,0.120286,2.135437,3122.119143,2,0,0.000000,inf,True,FCM,FCM_k5,...,0.200003,1.0,1.0,410.0,2022.0,True,1.0,1.0,1.0,1.0
4,0.120286,2.135437,3122.119143,2,0,0.000000,inf,True,FCM,FCM_k6,...,0.166669,1.0,1.0,415.0,2024.0,True,1.0,1.0,1.0,1.0
5,0.120286,2.135437,3122.119143,2,0,0.000000,inf,True,FCM,FCM_k7,...,0.142859,1.0,1.0,486.0,2026.0,True,1.0,1.0,1.0,1.0
6,0.120286,2.135437,3122.119143,2,0,0.000000,inf,True,FCM,FCM_k8,...,0.125002,1.0,1.0,413.0,2024.0,True,1.0,1.0,1.0,1.0
7,0.120286,2.135437,3122.119143,2,0,0.000000,inf,True,FCM,FCM_k9,...,0.111113,1.0,1.0,415.0,2023.0,True,1.0,1.0,1.0,1.0
8,0.120286,2.135437,3122.119143,2,0,0.000000,inf,True,FCM,FCM_k10,...,0.100002,1.0,1.0,405.0,2023.0,True,1.0,1.0,1.0,1.0


## 9. Ward sensitivity

Full k=2–10, matching K-Means and FCM. Ward and K-Means are fitted and evaluated on the same deterministic training sample. This population differs from selection-period metrics.

In [8]:
if not reuse_centroids: experiment.ward_sensitivity()
else: display(__import__('pandas').read_csv(experiment.out/'ward_sensitivity.csv'))

,silhouette,dbi,chi,occupied_clusters,min_cluster_n,min_cluster_fraction,imbalance_ratio,very_small_cluster_warning,method,k,evaluation_population,evaluation_n,silhouette_n,ward_vs_kmeans_ari,ward_vs_full_training_kmeans_ari,runtime_seconds
0,0.364438,1.302914,892.425762,2,426,0.10650,8.389671,False,Ward,2,same_deterministic_training_sample,4000,3000,0.567009,0.550405,1.737591
1,0.323760,1.679663,1006.695178,2,796,0.19900,4.025126,False,KMeans_same_sample,2,same_deterministic_training_sample,4000,3000,0.567009,0.550405,2.043269
2,0.184897,1.739230,883.594425,3,426,0.10650,5.213615,False,Ward,3,same_deterministic_training_sample,4000,3000,0.726725,0.723211,1.817462
3,0.197232,1.664719,947.012436,3,438,0.10950,4.972603,False,KMeans_same_sample,3,same_deterministic_training_sample,4000,3000,0.726725,0.723211,2.125498
4,0.182507,1.840463,766.607304,4,382,0.09550,4.814136,False,Ward,4,same_deterministic_training_sample,4000,3000,0.682167,0.694161,2.146830
5,0.207734,1.738695,841.451981,4,258,0.06450,7.821705,False,KMeans_same_sample,4,same_deterministic_training_sample,4000,3000,0.682167,0.694161,2.477310
6,0.164395,2.106236,659.674439,5,382,0.09550,4.814136,False,Ward,5,same_deterministic_training_sample,4000,3000,0.598697,0.638509,1.871919
7,0.161332,1.964950,733.785847,5,252,0.06300,5.841270,False,KMeans_same_sample,5,same_deterministic_training_sample,4000,3000,0.598697,0.638509,2.064374
8,0.167189,2.086699,601.686737,6,185,0.04625,9.940541,False,Ward,6,same_deterministic_training_sample,4000,3000,0.601419,0.589060,1.089557
9,0.167306,1.870412,670.188852,6,153,0.03825,9.666667,False,KMeans_same_sample,6,same_deterministic_training_sample,4000,3000,0.601419,0.589060,1.194199


## 10. Initialization stability

ARI between seed runs uses the identical full selection observation set. This measures initialization stability, separately from temporal persistence.

In [9]:
if not reuse_centroids: experiment.initialization_stability()
else: display(__import__('pandas').read_csv(experiment.out/'initialization_stability.csv').groupby(['method','k']).ari.agg(['mean','min']))

mean       min
method k                     
FCM    2   1.000000  1.000000
       3   1.000000  1.000000
       4   1.000000  1.000000
       5   1.000000  1.000000
       6   1.000000  1.000000
       7   1.000000  1.000000
       8   1.000000  1.000000
       9   1.000000  1.000000
       10  1.000000  1.000000
KMeans 2   1.000000  1.000000
       3   0.999946  0.999864
       4   0.997248  0.994546
       5   1.000000  1.000000
       6   0.997506  0.994391
       7   0.630323  0.395401
       8   0.659220  0.426858
       9   0.779545  0.472453
       10  0.997338  0.995898

## 10b. Density structural challenger

Six predeclared HDBSCAN configurations fit January–June on all 60 features; approximate_predict uses the fixed training tree. Coverage and non-noise scores are separated; matched-coverage K-Means comparisons are exported.

In [10]:
structural.density_challenger()

Verified and reused six executed HDBSCAN fits/predictions; regenerating physical/quality evidence.


## 10c. Redundancy sensitivity

Separate distance weighting for paired rotor/generator families, K-Means k=2–10. Main preprocessing and primary algorithm feature matrix remain frozen.

In [11]:
structural.redundancy_sensitivity()

## 11–14. Temporal, turbine, physical and quality audits

Consecutive one-hour pairs only, split boundaries preserved, original-unit profiles, turbine NMI and quality provenance. Ward profiles retain its identical training sample; flags do not enter distance space.

In [12]:
experiment.candidate_profiles()
structural.ward_profiles()

## 15–16. Comparison and evidence review

All tested candidates remain visible. Selection requires the measured profiles and trade-offs, not a ranking vote.

In [13]:
experiment.save_candidates()
experiment.selection_evidence()
display(__import__('pandas').read_csv(experiment.out/'hdbscan_comparison.csv'))


     model  silhouette      dbi         chi  min_cluster_fraction  seed_ari_min  persistence  turbine_nmi
 KMeans_k2    0.329538 1.856048 4071.275276              0.204704      1.000000     0.933531     0.000255
 KMeans_k3    0.159640 1.753498 3303.269423              0.103452      0.999864     0.890763     0.001649
 KMeans_k4    0.177315 1.781616 3049.039324              0.050190      0.994546     0.857224     0.002683
 KMeans_k5    0.180857 1.484015 3635.878678              0.000152      1.000000     0.857376     0.002746
 KMeans_k6    0.125096 1.753561 3193.322600              0.000152      0.994391     0.790565     0.002741
 KMeans_k7    0.135369 1.767890 2909.588900              0.000152      0.395401     0.748595     0.002899
 KMeans_k8    0.103539 1.908095 2661.287419              0.000152      0.426858     0.741644     0.002786
 KMeans_k9    0.114365 1.846789 2529.217558              0.000152      0.472453     0.718627     0.003146
KMeans_k10    0.118481 1.813001 2356.466710   

,model,method,min_cluster_size,min_samples,train_clusters,selection_clusters,train_noise_fraction,selection_noise_fraction,assigned_coverage_selection,evaluation_n,...,chi,occupied_clusters,min_cluster_n,min_cluster_fraction,imbalance_ratio,very_small_cluster_warning,persistence_all_including_noise,persistence_assigned_pairs_only,consecutive_assigned_pairs,noise_pair_fraction
0,HDBSCAN_c100_s10,HDBSCAN,100,10,3,3,0.208995,0.332625,0.667375,17592,...,1628.337110,3,303,0.017224,55.867987,False,0.781829,0.997893,14712,0.441203
1,HDBSCAN_c300_s10,HDBSCAN,300,10,3,3,0.208995,0.332625,0.667375,17592,...,1628.337110,3,303,0.017224,55.867987,False,0.781829,0.997893,14712,0.441203
2,HDBSCAN_c800_s10,HDBSCAN,800,10,2,2,0.203199,0.322951,0.677049,17847,...,3496.362116,2,919,0.051493,18.420022,False,0.774271,0.997176,14873,0.435088
3,HDBSCAN_c100_s30,HDBSCAN,100,30,3,3,0.258104,0.350531,0.649469,17120,...,1609.912024,3,217,0.012675,76.216590,False,0.780424,0.999156,14213,0.460156
4,HDBSCAN_c300_s30,HDBSCAN,300,30,2,2,0.256018,0.348141,0.651859,17183,...,2802.926750,2,644,0.037479,25.681677,False,0.777955,0.999087,14244,0.458979
5,HDBSCAN_c800_s30,HDBSCAN,800,30,2,2,0.256018,0.348141,0.651859,17183,...,2802.926750,2,644,0.037479,25.681677,False,0.777955,0.999087,14244,0.458979


## 17. Evidence-based selection and descriptive refit

The recorded decision is bound to the verified input hash. Full-year refit preserves preprocessing scaler parameters. On the initial execution only, the runner may wait for the research evidence review.

In [14]:
decision_path=experiment.out/'reviewed_selection.json'
if not decision_path.is_file() and os.environ.get('KELMARSH_WAIT_FOR_REVIEW')=='1':
    print('Measured candidate artifacts are ready; awaiting evidence-based configuration review.',flush=True)
    deadline=time.monotonic()+3600
    while not decision_path.is_file() and time.monotonic()<deadline:
        time.sleep(2)
if not decision_path.is_file():
    raise RuntimeError('Review measured candidate evidence and save reviewed_selection.json before final refit')
decision=json.loads(decision_path.read_text(encoding='utf-8'))
assert decision['input_sha256']==digest(experiment.input_dir/'windows_2022.csv.gz')
assert decision['reviewed_k_range']==list(range(2,11)), 'Decision must review the complete matching sweep'
assert decision['hdbscan_evidence_reviewed'], 'Review density coverage and physical profiles before final selection'
assert decision['hdbscan_comparison_sha256']==digest(experiment.out/'hdbscan_comparison.csv')
assert decision['centroid_comparison_sha256']==digest(experiment.out/'model_comparison.csv')
experiment.final_refit(decision['selected_model'],decision['rationale'])


Final descriptive refit: KMeans_k2; ARI to candidate=0.98129


## 18. Figures, paper narrative and export validation

Plots use original physical units where appropriate and 300 dpi. Candidate and refit results remain separate.

In [15]:
experiment.figures()
experiment.paper_report()
structural.append_report()
required=['input_manifest.json','environment_versions.json','feature_columns.json',
          'kmeans_sweep.csv','fcm_comparison.csv','ward_sensitivity.csv',
          'initialization_stability.csv','model_comparison.csv','cluster_sizes.csv',
          'cluster_profiles_median.csv','cluster_profiles_q25.csv','cluster_profiles_q75.csv',
          'cluster_distribution_by_turbine.csv','cluster_distribution_by_month.csv',
          'temporal_transitions.csv','quality_burden_by_cluster.csv','candidate_selection_report.md',
          'final_cluster_assignments.csv.gz','final_model_configuration.json']
assert all((experiment.out/name).is_file() for name in required)
print('COMPLETED:',experiment.out)


COMPLETED: C:\College\Sem5\DATMIN\Kelmarsh_SCADA_2022_4457\artifacts\kelmarsh_2022\clustering
